## Predicting Ski Conditions Across Vermont Using Historical Weather Data

Ned Culter and Maddy Smith  
Github: https://github.com/nedcut/ML-finalCSCI 0451  
5/19/2026



### Abstract

Having a ski mountain on campus is a great resource for Middlebury students, but ski conditions vary greatly, making the trip to the Snow Bowl feel either like a worthwhile commitment or a waste of time. This project will implement a multi-class ordinal logistic regression model using a time series of weather data, using factors like temperature, humidity, and precipitation to predict ski conditions on the mountains.

### Introduction


### Values Statement

We anticipate that this tool could be used by skiiers looking to determine whether a day on the mountain is worth their time, effort, and money. Resort-created ski forecasing services may skew towards an optimistic forecast, as more skiiers on the mountain results in higher profits. We hope that a third-party algorithm can give an objective prediction, allowing people to make more informed decisions.

It is possible that the resorts may be harmed by this software: if it tends to predict good days conservatively, it is possible that they will lose profit. However, we believe that hopefully helping skiiers identify good snowing days will help them find a passion for this sport, and stay on the slopes!

Having a ski mountain as a part of Middlebury is an integral part of campus culture. However, we are all busy students and a ski trip is a big committment. We want to create a tool that can help us anticipate conditions before we leave campus, so we can optimize our time on campus and on the mountain.

In [2]:
!git clone https://github.com/nedcut/ML-final.git
%cd ML-final
!pip install coral_pytorch

from pathlib import Path
import copy
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F

Cloning into 'ML-final'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 168 (delta 68), reused 80 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 11.71 MiB | 5.15 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/content/ML-final


### Data

We used two primary sources for this project. We got our response variables, or the ski grades, from Tony Crocker's "Best Snow" website. Crocker compiled and aggregated historical ski conditions for weekend ski conditions for the following resorts across Vermont: Killington, Sugarbush, Mad River, Stowe, Smugglers Notch, and Jay Peak. The data ranges from the 1990-2000 season, all the way up to the 2024-2025 season. Crocker's data gave each weekend an A/B/C/D grade, with each level corresponding to the following average conditions:

* A: Powder/packed powder on at least 50% of terrain. Generally 80+% of terrain open, but some leeway if most advanced runs and/or some of the tree skiing are open.
* B: 50-80% open with mostly packed powder, or 80+% open with less than 50% packed powder.
* C: At least one area 50+% open with hardpack/spring conditions.
* D: All areas less than 50% open.

To take Crocker's framework and extend it to a machine learning conditions model, we scraped the historical weather data from the different resorts from Open-Meteo Historical Weather API

We pulled comprehensive weather data for each report location from the Historical Weather API. We included factors like temperature, humidity, rainfall, snowfall, and snow depth.

It was unclear how Crocker handled his data to record his weekend grades. Crocker notes that "where available," he based his grade off the resort with the best conditions. He did not expand on his selection process, so we decided to aggregate the weekly data to two sets of variables. The first, denoted as "best_" is the weekly averaged data from the resort with the highest snowfall for the week. The second, denoted as "avg_" is the weekly averaged data between all of the locations available.

We also created variables of interest-- such as highest and lowest temperature, hours above freezing, and the mean, median, and mode temperature, rainfall, and snowfall.

Despite this system creating variables with high autocorrelation, we found that including both sets into our model resulted in the best performance.

With both of our data sources prepared, we then created a script that would match the weather data with its respective weekend from the "best snow" scraped data.

We are looking to use our data to predict the "grade" of the weekend. The explanatory variable list is shown below, as well as the head of our pandas data frame.

In [3]:
DATA_PATH = Path("data/processed/vt_condition_weather_aligned.csv")
df = pd.read_csv(DATA_PATH)

def weekly_features(df):
  """select numeric columns that start with 'avg_' or 'best_' and end with '_7d' """
  numeric_cols = [
      c for c in df.columns
      if (c.startswith("avg_") or c.startswith("best_"))
      and pd.api.types.is_numeric_dtype(df[c])
  ]
  return [c for c in numeric_cols if c.endswith("_7d")]

resp_vars = weekly_features(df)

for resp in resp_vars:
  print(resp)

df.head(15)

avg_hours_above_freezing_sum_7d
avg_hours_above_freezing_mean_7d
avg_hours_above_freezing_max_7d
avg_temperature_mean_7d
avg_temperature_min_7d
avg_temperature_max_7d
avg_humidity_mean_7d
avg_humidity_min_7d
avg_humidity_max_7d
avg_snow_depth_mean_7d
avg_snow_depth_min_7d
avg_snow_depth_max_7d
avg_wind_speed_mean_7d
avg_wind_speed_min_7d
avg_wind_speed_max_7d
avg_rain_sum_7d
avg_rain_max_7d
avg_snowfall_sum_7d
avg_snowfall_max_7d
best_hours_above_freezing_sum_7d
best_hours_above_freezing_mean_7d
best_hours_above_freezing_max_7d
best_temperature_mean_7d
best_temperature_min_7d
best_temperature_max_7d
best_humidity_mean_7d
best_humidity_min_7d
best_humidity_max_7d
best_snow_depth_mean_7d
best_snow_depth_min_7d
best_snow_depth_max_7d
best_wind_speed_mean_7d
best_wind_speed_min_7d
best_wind_speed_max_7d
best_rain_sum_7d
best_rain_max_7d
best_snowfall_sum_7d
best_snowfall_max_7d
best_location_count_7d


,season,season_start_year,season_end_year,season_week_index,season_chart_slot_index,month_index,month_name,month_number,calendar_year,month_week_slot,...,best_wind_speed_label_day,best_rain_sum_7d,best_rain_max_7d,best_rain_label_day,best_snowfall_sum_7d,best_snowfall_max_7d,best_snowfall_label_day,best_location_label_day,best_location_mode_7d,best_location_count_7d
0,2024-25,2024,2025,1,8,2,November,11,2024,3,...,8.525000,0.212,0.116,0.000,0.138,0.138,0.000,sugarbush,stowe,2
1,2024-25,2024,2025,2,9,2,November,11,2024,4,...,9.287500,0.881,0.542,0.100,4.055,3.033,1.022,killington,sugarbush,2
2,2024-25,2024,2025,3,10,2,November,11,2024,5,...,10.908333,0.568,0.421,0.000,3.948,1.875,0.000,killington,jay_peak,2
3,2024-25,2024,2025,4,11,3,December,12,2024,1,...,4.866667,0.004,0.004,0.000,4.670,1.740,0.333,jay_peak,jay_peak,2
4,2024-25,2024,2025,5,12,3,December,12,2024,2,...,2.325000,1.213,0.795,0.000,5.683,2.674,0.000,stowe,stowe,2
5,2024-25,2024,2025,6,13,3,December,12,2024,3,...,6.483333,0.198,0.186,0.004,3.374,1.710,0.641,stowe,stowe,1
6,2024-25,2024,2025,7,14,3,December,12,2024,4,...,2.791667,0.000,0.000,0.000,3.832,3.748,0.000,stowe,stowe,1
7,2024-25,2024,2025,8,16,4,January,1,2025,1,...,11.329167,0.629,0.415,0.000,4.720,2.233,0.997,jay_peak,stowe,2
8,2024-25,2024,2025,9,17,4,January,1,2025,2,...,2.400000,0.000,0.000,0.000,5.446,1.517,0.801,jay_peak,jay_peak,2
9,2024-25,2024,2025,10,18,4,January,1,2025,3,...,8.987500,0.004,0.004,0.004,3.820,1.657,0.112,stowe,jay_peak,2


From there, we split our data into training and testing seasons, and made our data into tensors.

In [4]:
DATA_PATH = Path("data/processed/vt_condition_weather_aligned.csv")
CHECKPOINT_PATH = Path("data/processed/best_ski_coral.pt")
SEED = 3

TEST_SEASONS = 5
VAL_SEASONS = 3

def make_split(df):
    """split data into train, val, test based on season, using the last seasons for test and val"""
    seasons = sorted(df["season"].unique(), key=lambda s: int(str(s).split("-")[0]))
    test_seasons = seasons[-TEST_SEASONS:]
    val_start = -(TEST_SEASONS + VAL_SEASONS)
    val_seasons = seasons[val_start:-TEST_SEASONS]
    train_seasons = seasons[:val_start]

    train_df = df[df["season"].isin(train_seasons)].reset_index(drop=True)
    val_df = df[df["season"].isin(val_seasons)].reset_index(drop=True)
    test_df = df[df["season"].isin(test_seasons)].reset_index(drop=True)

    print(f"train seasons:  {train_seasons[0]} through {train_seasons[-1]} ({len(train_df)} rows)")
    print(f"val seasons:  {val_seasons[0]} through {val_seasons[-1]} ({len(val_df)} rows)")
    print(f"test seasons: {test_seasons[0]} through {test_seasons[-1]} ({len(test_df)} rows)")
    return train_df, val_df, test_df


def weekly_features(df):
    """select numeric columns that start with 'avg_' or 'best_' and end with '_7d' """
    numeric_cols = [
        c for c in df.columns
        if (c.startswith("avg_") or c.startswith("best_"))
        and pd.api.types.is_numeric_dtype(df[c])
    ]
    return [c for c in numeric_cols if c.endswith("_7d")]


def make_tensors(train_df, val_df, test_df, feature_cols):
    """convert dataframes to tensors, normalizing features based on train_df statistics"""
    train_x = train_df[feature_cols].to_numpy(dtype="float32")
    val_x = val_df[feature_cols].to_numpy(dtype="float32")
    test_x = test_df[feature_cols].to_numpy(dtype="float32")

    mu = train_x.mean(axis=0, keepdims=True)
    sigma = train_x.std(axis=0, keepdims=True)
    sigma[sigma == 0] = 1

    train_x = (train_x - mu) / sigma
    val_x = (val_x - mu) / sigma
    test_x = (test_x - mu) / sigma

    return (
        torch.tensor(train_x, dtype=torch.float32),
        torch.tensor(train_df["y"].to_numpy(), dtype=torch.long),
        torch.tensor(val_x, dtype=torch.float32),
        torch.tensor(val_df["y"].to_numpy(), dtype=torch.long),
        torch.tensor(test_x, dtype=torch.float32),
        torch.tensor(test_df["y"].to_numpy(), dtype=torch.long),
    )

## Approach
### Model Considerations

We will need to implement a model that is able to handle the ordinal nature of the data. We need a model that is able to recognize that the ordering of the labels is significant, or that a D day is worse than a C day, which is worse than a B day, which is worse than an A day.

Using a loss function like cross entropy loss only recognizes that the model is making an incorrect prediction. It does not recognize that there are different "severity levels" of mistakes, or that classifying a B day as an A day is better than classifying a B day as a D day.

To do this, we implemented a CORAL Layer within a Neural Network, which implements a form of cross entropy that uses logits.

This report will contain our advanced model, while comparing it to a baseline neural network that implements cross entropy to demonstrate the importance of implementing a model that respects ordinal structure for target labels with meaningful order.

Furthermore, we implemented a cost matrix to assess the performance of both models. Along with assessing accuracy and F1 scores, we developed the following cost matrix to get an average "score" of the model, which will be discussed more in later sections.


### CORAL Layer and Ordinal Models

Ordinal Regression models turn classfications into multiple binary classification thresholds.

For our 4 classes (A, B, C, D), the model computes the four following tasks.

* 1: What is the probability of the data point being greater than grade D?
  * P(Grade > D)
* 2: What is the probability of the data point being greater than grade C?
  * P(Grade > C)
* 3: What is the probability of the data point being greater than grade D?
  * P(Grade > B)

The model then learns probability thresholds to predict the four separate classes.

A large limitation of ordinal models is that the thresholds are not ofen monotonic.

For example, an ordinal regession model may compute P(Grade > B) > P(Grade > C), which is nonsensical.

Implementing a CORAL layer guarantees monotonic decreases in probability calculations, in contrast to a ordinal logistic loss model (Figure 1).


<img src = 'https://drive.google.com/uc?id=1MUb0pOz1TBkT1eMQ61fGdwUp8HX4Z1at' height = '400'>

(Figure 1)



### CORAL Layer Implementation

Adding a CORAL Layer to a ordinal regression model enforces rank consistency (CITATION: Rank consistent ordinal regression for neural networks with application to age estimation)

In CORAL, all $K-1$ binary decision share the same weights W, but each binary decision gets its own bias term, $b_k$.

Here, the bias is initialized as [1, 0, -1]. During the forward pass, the bias term is updated to: [$w^Tx_i + b_1, w^Tx_i + b_2, w^Tx_i + b_3$]. Since the weights are shared and the biases are ordered, this ensures monoticity. The probability for the $k^{th}$ decision is determined by:

$\sigma(w^Tx_i  + b_k)$

Since the sigmoid is a monotonically increasing function, ordered biases implies ordered probabilities per binary decision, guarunteeing rank-consistent decisions and that:

P(Grade > D) > P(Grade > C) > P(Grade > B)


The CORAL class is implemented below.


In [5]:
class CoralLayer(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
        self.shared_weight = nn.Linear(n_features, 1, bias=False)
        self.bias = nn.Parameter(torch.linspace(1.0, -1.0, n_classes - 1))

    def forward(self, x):
        return self.shared_weight(x) + self.bias

To make this a properly ordinal model, we must create functions that will calculate three binary classification tasks.

Our coral_targets function calculates the cumulative probabilities P(Grade > D), P(Grade > C), P(Grade > B).

From there, the loss is calculated as a weighted cross-entropy of the 3 binary classifiers. It calculates the cross entropy between the logits calculated by our neural network and the response grade, which has been transformed by our coral_targets.

In [6]:
GRADE_ORDER = ["D", "C", "B", "A"]
GRADE_TO_Y = {grade: i for i, grade in enumerate(GRADE_ORDER)} # D=0, C=1, B=2, A=3
Y_TO_GRADE = {i: grade for grade, i in GRADE_TO_Y.items()}     # 0=D, 1=C, 2=B, 3=A
EVAL_GRADE_ORDER = ["A", "B", "C", "D"]


def coral_targets(y):
    thresholds = torch.arange(len(GRADE_ORDER) - 1, device=y.device)
    return (y[:, None] > thresholds[None, :]).float()


def coral_loss(logits, y):
    return F.binary_cross_entropy_with_logits(logits, coral_targets(y))

### Model Architecture

We implemented a simple Multilayer Perceptron (MLP), with a CORAL layer as the final layer.

Our MLP is structured as follows:

1. A Linear layer of size (n_features x hidden)
2. A non-linear ReLU activation function
3. A Dropout layer that will set a random fraction of inputs to 0, which will reduce overfitting
4. A CORAL layer that updates decision boundaries of size (HIDDEN, n_classes)

Our model class is shown below:

In [7]:
class SkiCoral(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(n_features, HIDDEN),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
        )
        self.coral = CoralLayer(HIDDEN, len(GRADE_ORDER))

    def forward(self, x):
        return self.coral(self.features(x))

### Training

We ran a standard training loop on our model. We initialized 500 epochs with a learning rate of 0.006, but implemented a termination condition if our model converged before the training loop ended. We saved the model with the lowest loss, and used that model to create our predictions.



In [8]:
HIDDEN = 16
DROPOUT = 0.30
LR = 0.006
WEIGHT_DECAY = 0.0
MAX_EPOCHS = 500
PATIENCE = 50
MIN_DELTA = 1e-5
PRED_THRESHOLD = 0.60


def train(model, x_train, y_train, x_val, y_val):
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_val_loss = float("inf")
    best_epoch = 0
    best_state = copy.deepcopy(model.state_dict())
    bad_checks = 0 # number of consecutive epochs without improvement

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        opt.zero_grad()
        loss = coral_loss(model(x_train), y_train)
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = float(coral_loss(model(x_val), y_val))

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_checks = 0
        else:
            bad_checks += 1

        if bad_checks >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return best_epoch, epoch, best_val_loss

def save_checkpoint(model, feature_cols, best_epoch, stopped_epoch, best_val_loss):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "feature_cols": feature_cols,
        "grade_order": GRADE_ORDER,
        "config": {
            "hidden": HIDDEN,
            "dropout": DROPOUT,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "pred_threshold": PRED_THRESHOLD,
            "seed": SEED,
            "test_seasons": TEST_SEASONS,
            "val_seasons": VAL_SEASONS,
            "best_epoch": best_epoch,
            "stopped_epoch": stopped_epoch,
            "best_val_loss": best_val_loss,
        },
    }
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(checkpoint, CHECKPOINT_PATH)
    print(f"saved checkpoint: {CHECKPOINT_PATH}")

### Evaluation
We used multiple metrics to determine the performance of our model. We looked at standard measures such as loss, F1 scores, and accuracy between the training, validation, and testing sets.

We also implemented two unique performance metrics. First, we calculated the proportion of predictions that were within 1 grade of the correct grade, which we classified as an "acceptable" prediction.

To formalize that idea further, we also implemented a cost matrix. The cost matrix assigns penalties to each misclassification.

| Actual \ Predicted | D   | C   | B   | A   |
|--------------------|-----|-----|-----|-----|
| D                  | 0.0 | 0.5 | 2.0 | 4.0 |
| C                  | 0.5 | 0.0 | 0.5 | 2.0 |
| B                  | 2.0 | 0.5 | 0.0 | 1.0 |
| A                  | 4.0 | 2.0 | 0.5 | 0.0 |


This assigns higher penalties to predictions that are further from the truth, giving us a better understanding of our model's ability to make acceptable mistakes.

In [9]:
COST_MATRIX = torch.tensor([
#pred D    C    B    A
    [0.0, 0.5, 2.0, 4.0], # actual D
    [0.5, 0.0, 0.5, 2.0], # actual C
    [2.0, 0.5, 0.0, 1.0], # actual B
    [4.0, 2.0, 0.5, 0.0], # actual A
])

def macro_f1(preds, y):
    scores = []
    for cls in range(4):
        tp = ((preds == cls) & (y == cls)).sum().item()
        fp = ((preds == cls) & (y != cls)).sum().item()
        fn = ((preds != cls) & (y == cls)).sum().item()
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        scores.append(2 * precision * recall / (precision + recall) if precision + recall else 0.0)
    return float(np.mean(scores))

def coral_preds(logits):
    return (torch.sigmoid(logits) > PRED_THRESHOLD).sum(dim=1)

def evaluate(model, x, y):
    model.eval()
    with torch.no_grad():
        preds = coral_preds(model(x))
        cost_matrix = COST_MATRIX.to(y.device)
    return {
        "accuracy": (preds == y).float().mean().item(),
        # if the target is only 1 grade away from prediction (eg predicted B but it's actually A)
        "within_one": ((preds - y).abs() <= 1).float().mean().item(),
        "macro_f1": macro_f1(preds, y),
        "avg_cost": cost_matrix[y, preds].mean().item(),
        "preds": preds,
    }

def print_metrics(name, metrics):
    print(
        f"{name:>4}: "
        f"accuracy={metrics['accuracy']:.3f}, "
        f"macro_f1={metrics['macro_f1']:.3f}, "
        f"within_one={metrics['within_one']:.3f}, "
        f"avg_cost={metrics['avg_cost']:.3f}"
    )

### Evaluation

Finally, we run our model on our training, validation, and testing data sets

In [10]:
if __name__ == "__main__":
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    df = pd.read_csv(DATA_PATH)
    df["y"] = df["grade"].map(GRADE_TO_Y)

    train_df, val_df, test_df = make_split(df)
    feature_cols = weekly_features(df)
    print(f"features: {len(feature_cols)}")

    x_train, y_train, x_val, y_val, x_test, y_test = make_tensors(
        train_df,
        val_df,
        test_df,
        feature_cols,
    )

    model = SkiCoral(x_train.shape[1])
    best_epoch, stopped_epoch, best_val_loss = train(model, x_train, y_train, x_val, y_val)

    print(f"best epoch: {best_epoch}")
    print(f"stopped epoch: {stopped_epoch}")
    print(f"best val loss: {best_val_loss:.4f}")
    save_checkpoint(model, feature_cols, best_epoch, stopped_epoch, best_val_loss)

    train_metrics = evaluate(model, x_train, y_train)
    val_metrics = evaluate(model, x_val, y_val)
    test_metrics = evaluate(model, x_test, y_test)

    print_metrics("train", train_metrics)
    print_metrics("val", val_metrics)
    print_metrics("test", test_metrics)

    confusion = pd.crosstab(
        pd.Series(test_df["y"].map(Y_TO_GRADE), name="actual"),
        pd.Series(pd.Series(test_metrics["preds"].numpy()).map(Y_TO_GRADE), name="predicted"),
    ).reindex(index=EVAL_GRADE_ORDER, columns=EVAL_GRADE_ORDER, fill_value=0)

    cost_df = pd.DataFrame(COST_MATRIX.numpy(), index=GRADE_ORDER, columns=GRADE_ORDER)
    cost_df = cost_df.reindex(index=EVAL_GRADE_ORDER, columns=EVAL_GRADE_ORDER)
    cost_contribution = confusion * cost_df

train seasons:  1999-00 through 2016-17 (510 rows)
val seasons:  2017-18 through 2019-20 (90 rows)
test seasons: 2020-21 through 2024-25 (141 rows)
features: 39
best epoch: 335
stopped epoch: 385
best val loss: 0.2743
saved checkpoint: data/processed/best_ski_coral.pt
train: accuracy=0.729, macro_f1=0.705, within_one=0.992, avg_cost=0.161
 val: accuracy=0.711, macro_f1=0.700, within_one=0.989, avg_cost=0.167
test: accuracy=0.631, macro_f1=0.538, within_one=0.957, avg_cost=0.255


With training, validation, and testing accuracies of 0.73, 0.71, and 0.63, respectively, this model is properly capturing the underluying signal of the data and is not severely overfitting on the training data.

Furthermore, the training, validation, and testing data have a "within one" score of 0.99, 0.99, and 0.96, respectively. The three data sets also have average costs of 0.16, 0.17, and 0.25.

In [11]:
print("\ntest confusion matrix")
print(confusion)

print("\ntest cost contribution matrix")
print(cost_contribution)
print(f"test total cost: {cost_contribution.to_numpy().sum():.1f}")


test confusion matrix
predicted   A   B   C   D
actual                   
A          11  14   3   0
B           2   8  11   1
C           1   4  13  13
D           0   1   2  57

test cost contribution matrix
predicted    A    B    C    D
actual                       
A          0.0  7.0  6.0  0.0
B          2.0  0.0  5.5  2.0
C          2.0  2.0  0.0  6.5
D          0.0  2.0  1.0  0.0
test total cost: 36.0


Looking at the confusion matrix and the cost contribution for our test set, we see that although it slightly underperforms our training and validation data, it is still remarkably useful for our task of predicting ski conditions.

There are no A -> D or D -> A unacceptable misclassifications, and the majority of the cost penalty is coming from the acceptable "one-off" predictions. We therefore conclude that this model does a satasfactory job at predicting ski conditions, and is able to properly handle the ordinal structure of the classification.

## Baseline Models

We will compare our CORAL MLP to a multinomial logistic regression model that does not innately handle ordinal reponse labels. We will include the same MLP architecture, but will calculate our loss with a standard cross entropy loss function.

In [24]:
class LogisticRegression(nn.Module):
  def __init__(self, n_features, n_classes):
    super().__init__()

    self.pipeline = nn.Sequential(
      nn.Linear(n_features, HIDDEN),
      nn.ReLU(),
      nn.Dropout(DROPOUT),
      nn.Linear(HIDDEN, n_classes)
    )

  def forward(self, X):
    return self.pipeline(X)

def train_baseline(model, x_train, y_train, x_val, y_val):
  opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
  best_val_loss = float("inf")
  best_epoch = 0
  best_state = copy.deepcopy(model.state_dict())
  bad_checks = 0 # number of consecutive epochs without improvement
  loss = nn.CrossEntropyLoss()
  losses = []

  for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    opt.zero_grad()
    scores = model(x_train)
    loss_train = loss(scores, y_train)
    losses.append(loss_train.item())
    loss_train.backward()
    opt.step()

    model.eval()
    with torch.no_grad():
      scores_eval = model(x_val)
      val_loss = loss(scores_eval, y_val)

    if val_loss < best_val_loss - MIN_DELTA:
      best_val_loss = val_loss
      best_epoch = epoch
      best_state = copy.deepcopy(model.state_dict())
      bad_checks = 0
    else:
      bad_checks += 1

    if bad_checks >= PATIENCE:
      break

  model.load_state_dict(best_state)
  return best_epoch, epoch, best_val_loss

def predict(model, X):
    with torch.no_grad():
        scores = model(X)
        probs = torch.softmax(scores, dim=1)
        preds = probs.argmax(dim=1)
    return preds, probs

def evaluate(model, x, y):
    model.eval()
    with torch.no_grad():
        preds, probs = predict(model, x)
        cost_matrix = COST_MATRIX.to(y.device)
    return {
        "accuracy": (preds == y).float().mean().item(),
        # if the target is only 1 grade away from prediction (eg predicted B but it's actually A)
        "within_one": ((preds - y).abs() <= 1).float().mean().item(),
        "macro_f1": macro_f1(preds, y),
        "avg_cost": cost_matrix[y, preds].mean().item(),
        "preds": preds,
    }

if __name__ == "__main__":

  model = LogisticRegression(x_train.shape[1], len(GRADE_ORDER))
  best_epoch, stopped_epoch, best_val_loss = train_baseline(model, x_train, y_train, x_val, y_val)

  print(f"best epoch: {best_epoch}")
  print(f"stopped epoch: {stopped_epoch}")
  print(f"best val loss: {best_val_loss:.4f}")
  save_checkpoint(model, feature_cols, best_epoch, stopped_epoch, best_val_loss)

  train_metrics = evaluate(model, x_train, y_train)
  val_metrics = evaluate(model, x_val, y_val)
  test_metrics = evaluate(model, x_test, y_test)

  print_metrics("train", train_metrics)
  print_metrics("val", val_metrics)
  print_metrics("test", test_metrics)

  confusion = pd.crosstab(
    pd.Series(test_df["y"].map(Y_TO_GRADE), name="actual"),
    pd.Series(pd.Series(test_metrics["preds"].numpy()).map(Y_TO_GRADE), name="predicted"),
  ).reindex(index=EVAL_GRADE_ORDER, columns=EVAL_GRADE_ORDER, fill_value=0)

  cost_df = pd.DataFrame(COST_MATRIX.numpy(), index=GRADE_ORDER, columns=GRADE_ORDER)
  cost_df = cost_df.reindex(index=EVAL_GRADE_ORDER, columns=EVAL_GRADE_ORDER)
  cost_contribution = confusion * cost_df

best epoch: 86
stopped epoch: 136
best val loss: 0.7871
saved checkpoint: data/processed/best_ski_coral.pt
train: accuracy=0.690, macro_f1=0.652, within_one=0.975, avg_cost=0.228
 val: accuracy=0.722, macro_f1=0.708, within_one=0.944, avg_cost=0.233
test: accuracy=0.667, macro_f1=0.613, within_one=0.957, avg_cost=0.252


Looking at our baseline model, our accuracy is reasonably similar to our model. However, we see a slight decrease in the "within_one" metric in both the training and the validation set. On top of that, the average cost is higher for the train and validation set as well.

Although our testing data performs very similarly to the test data from our CORAL layer model, looking at the actual predictions tells a slightly different story.

In [25]:
  print("\ntest confusion matrix")
  print(confusion)

  print("\ntest cost contribution matrix")
  print(cost_contribution)
  print(f"test total cost: {cost_contribution.to_numpy().sum():.1f}")


test confusion matrix
predicted   A   B   C   D
actual                   
A          14  10   3   1
B           2  13   6   1
C           1   6  15   9
D           0   0   8  52

test cost contribution matrix
predicted    A    B    C    D
actual                       
A          0.0  5.0  6.0  4.0
B          2.0  0.0  3.0  2.0
C          2.0  3.0  0.0  4.5
D          0.0  0.0  4.0  0.0
test total cost: 35.5


Looking at our confusion and cost contribution matrix, you can see that there is a weekend classified as a D that is actually an A day. Our CORAL layer has 0 instances of that unacceptable classification. You could imagine that a larger dataset would increase the number of unnacceptable classifications, making the model not reliable for skiiers, versus our CORAL layer which avoided making any of those unnacceptable errors.